In [ ]:
import math, re, sys, json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

import pennylane as qml
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [ ]:
# ----------------------------
# Fix random seeds for reproducibility
# ----------------------------
np.random.seed(42)
torch.manual_seed(42)


# ----------------------------
# Config
# ----------------------------
MIN_RATING = 0
MAX_RATING = 5
NUM_RATINGS = MAX_RATING - MIN_RATING + 1  # e.g., 6
# default small LLM for local tests
# LLM_NAME = "distilgpt2"
LLM_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
# PennyLane device shots
QL_SHOTS = 512

In [ ]:
# ----------------------------
# Utilities
# ----------------------------
def n_qubits_for_num_ratings(num_ratings):
    return math.ceil(math.log2(num_ratings))

N_QUBITS = n_qubits_for_num_ratings(NUM_RATINGS)
print(f"Using {N_QUBITS} qubits (can represent {2**N_QUBITS} states) for {NUM_RATINGS} ratings")

def rating_to_index(r):
    """Map rating in [MIN_RATING,MAX_RATING] to index 0..NUM_RATINGS-1"""
    idx = int(round(float(r))) - MIN_RATING
    idx = max(0, min(NUM_RATINGS - 1, idx))
    return idx

def index_to_rating(idx):
    return MIN_RATING + int(idx)

def index_to_bits(idx, n_qubits=N_QUBITS):
    b = format(int(idx), f"0{n_qubits}b")
    return [int(ch) for ch in b]

def bits_to_index(bits):
    return int("".join(str(b) for b in bits), 2)

In [ ]:
# ----------------------------
# LLM helper
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(LLM_NAME)
# If model doesn't have pad_token, avoid warnings by setting
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# def make_prompt(row):
#     # Keep prompt explicit and directive to get a single number
#     return (
#         f"User: sex={row.get('sex','?')}, age={row.get('age','?')}, country={row.get('Country','?')}, mood={row.get('mood','?')}\n"
#         f"Movie: title={row.get('Movie_Name','?')}, director={row.get('Director','?')}, genres={row.get('Genre1','?')}/{row.get('Genre2','?')}\n"
#         "Question: On a scale from 0 to 5 (inclusive), what rating would the user give this movie? "
#         "Answer with only a single number between 0 and 5 (e.g., 3 or 4.5).\nAnswer: "
#     )

def make_prompt(row):
        return (
            f"User ID: {row.get('user_id', '?')}\n"
            f"Business ID: {row.get('business_id', '?')}\n"
            f"Review: {row.get('text', '')}\n"
            "Question: Based on the review above, what star rating (0 to 5) would the user give this business? "
            "Answer with only a single number between 0 and 5 (e.g., 3 or 4.5).\nAnswer: "
        )


# def extract_rating_from_text(text):
#     # find the first occurrence of a number between 0 and 5 (allow decimal)
#     m = re.search(r'(?<!\d)([0-5](?:\.\d+)?)', text)
#     if m:
#         val = float(m.group(1))
#         return max(MIN_RATING, min(MAX_RATING, val))
#     # fallback: return midpoint
#     return float((MIN_RATING + MAX_RATING) / 2.0)

# def llm_predict_row(row, max_new_tokens=20):
#     prompt = make_prompt(row)
#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
#     out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
#     decoded = tokenizer.decode(out[0], skip_special_tokens=True)
#     # sometimes model repeats prompt; try to take the trailing part after "Answer"
#     if "Answer" in decoded:
#         decoded = decoded.split("Answer", 1)[1]
#     rating = extract_rating_from_text(decoded)
#     return rating, decoded.strip()
def extract_rating_from_text(text):
    # Try to find the last "Answer" block then search for number there (safer)
    # Normalize separators
    text = text.replace("\n", " ").strip()
    # If there is 'Answer' or 'Answer:' take the substring after the last occurrence
    if "Answer" in text:
        parts = re.split(r'Answer[:\s]*', text, flags=re.IGNORECASE)
        candidate = parts[-1] if parts[-1].strip() != "" else text
    else:
        candidate = text
    # Now search for a number between 0 and 5 (allow decimals)
    m = re.search(r'(?<!\d)([0-5](?:\.\d+)?)(?!\d)', candidate)
    if m:
        val = float(m.group(1))
        return max(MIN_RATING, min(MAX_RATING, val))
    # fallback: search the whole text (last chance)
    m2 = re.search(r'([0-5](?:\.\d+)?)', text)
    if m2:
        return max(MIN_RATING, min(MAX_RATING, float(m2.group(1))))
    # final fallback: midpoint
    return float((MIN_RATING + MAX_RATING) / 2.0)

def llm_predict_row(row, max_new_tokens=10):
    prompt = make_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    # Ensure model and inputs on same device
    device = next(model.parameters()).device
    inputs = {k:v.to(device) for k,v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # debug: return decoded for inspection
    rating = extract_rating_from_text(decoded)
    return rating, decoded.strip()

In [ ]:
# Calcul automatique du nombre d’itérations Grover, j'ai ajouté une fonction utilitaire pour calculer k optimal :
def optimal_grover_iterations(n_qubits, n_marked=1):
    N = 2 ** n_qubits
    M = max(1, n_marked)
    k = int(math.floor((math.pi / 4.0) * math.sqrt(N / M)))
    return max(1, k)

In [ ]:
def safe_multi_controlled_x(controls, target):
    # Try to call the native op; if not available, implement a simple decomposition for small n
    try:
        qml.MultiControlledX(wires=controls + [target])
    except Exception as e:
        # Decomposition for 1 or 2 controls (toy fallback)
        if len(controls) == 1:
            qml.CNOT(wires=[controls[0], target])
        elif len(controls) == 2:
            # Toffoli
            qml.Toffoli(wires=[controls[0], controls[1], target])
        else:
            raise RuntimeError("MultiControlledX not available and no fallback implemented for >2 controls.")


In [ ]:
# ----------------------------
# Quantum: Oracle & Diffusion (Grover-like)
# ----------------------------
def apply_oracle_ops(target_bits):
    # returns a python function that applies the oracle marking target_bits with a phase flip
    def oracle():
        # flip qubits where bit == 0
        for i, b in enumerate(target_bits):
            if b == 0:
                qml.PauliX(wires=i)
        # multi-controlled Z via H on target & MultiControlledX
        if len(target_bits) == 1:
            qml.PauliZ(wires=0)
        else:
            target = len(target_bits) - 1
            qml.Hadamard(wires=target)
            controls = list(range(0, len(target_bits) - 1))
            safe_multi_controlled_x(controls, target)
            qml.Hadamard(wires=target)
        # undo flips
        for i, b in enumerate(target_bits):
            if b == 0:
                qml.PauliX(wires=i)
    return oracle

In [ ]:
def diffusion_ops(n_qubits):
    def diffuse():
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
        for i in range(n_qubits):
            qml.PauliX(wires=i)
        if n_qubits == 1:
            qml.PauliZ(wires=0)
        else:
            target = n_qubits - 1
            qml.Hadamard(wires=target)
            controls = list(range(0, n_qubits - 1))
            safe_multi_controlled_x(controls, target)
            qml.Hadamard(wires=target)
        for i in range(n_qubits):
            qml.PauliX(wires=i)
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
    return diffuse

In [ ]:
def build_grover_qnode(n_qubits, shots=QL_SHOTS, interface='autograd'):
    dev = qml.device("default.qubit", wires=n_qubits, shots=shots)
    def make_qnode(k_iter):
        @qml.qnode(dev, interface=interface)
        def qnode(target_bits):
            # prepare uniform superposition
            for i in range(n_qubits):
                qml.Hadamard(wires=i)
            oracle_fn = apply_oracle_ops(target_bits)
            diffuse_fn = diffusion_ops(n_qubits)
            for _ in range(k_iter):
                oracle_fn()
                diffuse_fn()
            # sample computational basis on all wires
            return qml.sample(wires=list(range(n_qubits)))
        return qnode
    return make_qnode


In [ ]:
# === Figure Grover qnode Yelp -> PNG (basé sur build_grover_qnode) ===
import pennylane as qml
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Helpers (si tu n'as pas déjà ces fonctions dans ton notebook)
# ---------------------------------------------------------
def diffusion_ops(n_qubits):
    """Diffusion standard (inversion about the mean) sur n_qubits."""
    def _diffuse():
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.PauliX(wires=i)

        # multi-controlled X (approx du multi-controlled Z avec X sur dernier)
        control_wires = list(range(n_qubits - 1))
        target_wire = n_qubits - 1
        qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

        for i in range(n_qubits):
            qml.PauliX(wires=i)
            qml.Hadamard(wires=i)
    return _diffuse


def apply_oracle_ops(target_bits):
    """
    Oracle de Grover qui marque l'état |target_bits>.
    target_bits: liste/array de 0/1 de longueur n_qubits.
    """
    n_qubits = len(target_bits)

    def _oracle():
        # amener l'état cible vers |111...> avec X sur les bits à 0
        for i, b in enumerate(target_bits):
            if int(b) == 0:
                qml.PauliX(wires=i)

        control_wires = list(range(n_qubits - 1))
        target_wire = n_qubits - 1
        qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

        # revenir au basis standard
        for i, b in enumerate(target_bits):
            if int(b) == 0:
                qml.PauliX(wires=i)

    return _oracle


# ---------------------------------------------------------
# Ton builder (adapté pour le rendu)
# ---------------------------------------------------------
def build_grover_qnode_for_drawing(n_qubits, interface="autograd"):
    """
    Version 'drawing' : device analytique (shots=None) + expval,
    pour un rendu clean avec qml.draw_mpl.
    """
    dev_draw = qml.device("default.qubit", wires=n_qubits, shots=None)

    def make_qnode(k_iter):
        @qml.qnode(dev_draw, interface=interface)
        def qnode(target_bits):
            # superposition uniforme
            for i in range(n_qubits):
                qml.Hadamard(wires=i)

            oracle_fn = apply_oracle_ops(target_bits)
            diffuse_fn = diffusion_ops(n_qubits)

            for _ in range(k_iter):
                oracle_fn()
                diffuse_fn()

            # pour le dessin : on met des mesures expval (plutôt que sample)
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        return qnode

    return make_qnode


# ---------------------------------------------------------
# Génération de la figure
# ---------------------------------------------------------
# Paramètres Yelp (ex: 3 qubits -> 8 états >= 5 ratings)
N_QUBITS = 3

# Exemple: cible = rating 5 -> index 4 -> bits "100" (selon ton mapping)
# IMPORTANT: ici target_bits est un motif binaire, pas un rating directement.
# Mets celui qui correspond à ton mapping Yelp dans hybrid_predict_row.
target_bits = [1, 0, 0]

k_iter = 1  # comme dans ton éval Yelp (grover_iters=1)

qnode_maker_draw = build_grover_qnode_for_drawing(N_QUBITS)
qnode_draw = qnode_maker_draw(k_iter)

drawer = qml.draw_mpl(qnode_draw)
fig, ax = drawer(target_bits)

fig.set_size_inches(14, 3.2)
fig.savefig("grover_yelp_circuit.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("✅ Saved: grover_yelp_circuit.png")


In [ ]:
# ----------------------------
# Run Grover and decode predicted rating (most frequent bitstring)
# ----------------------------
# def run_grover_get_rating(qnode, target_bits, k_iter=1):
#     # qnode returns array shape (shots, n_qubits) typically
#     samples = qnode(target_bits)  # may be numpy array
#     # Convert samples to bitstrings
#     samples_arr = np.array(samples)
#     # If shape is (n_qubits, shots) (older API), transpose
#     if samples_arr.ndim == 2 and samples_arr.shape[0] == N_QUBITS and samples_arr.shape[1] == qnode.device.num_shots:
#         samples_arr = samples_arr.T
#     # Now samples_arr is (shots, n_qubits) with values 0/1 typically
#     bitstrings = ["".join(str(int(b)) for b in row) for row in samples_arr]
#     counts = Counter(bitstrings)
#     top_bits, top_count = counts.most_common(1)[0]
#     predicted_index = bits_to_index([int(ch) for ch in top_bits])
#     # if predicted index maps beyond NUM_RATINGS (because 2^n > NUM_RATINGS), clamp:
#     predicted_index = min(predicted_index, NUM_RATINGS - 1)
#     predicted_rating = index_to_rating(predicted_index)
#     return predicted_rating, counts
# je l'ai améliorer en ajoutant  aussi la distribution counts triée et la probabilité du top
def run_grover_get_rating(qnode, target_bits, k_iter=1):
    samples = qnode(target_bits)
    samples_arr = np.array(samples)
    # handle shape variants
    if samples_arr.ndim == 2 and samples_arr.shape[0] == N_QUBITS and samples_arr.shape[1] == qnode.device.shots:
        samples_arr = samples_arr.T
    if samples_arr.ndim == 1:
        # single sample case -> expand
        samples_arr = np.expand_dims(samples_arr, axis=0)
    bitstrings = ["".join(str(int(b)) for b in row) for row in samples_arr]
    counts = Counter(bitstrings)
    total = sum(counts.values())
    top_bits, top_count = counts.most_common(1)[0]
    predicted_index = bits_to_index([int(ch) for ch in top_bits])
    predicted_index = min(predicted_index, NUM_RATINGS - 1)
    predicted_rating = index_to_rating(predicted_index)
    # also produce normalized distribution dictionary
    distr = {k: v / total for k, v in counts.items()}
    return predicted_rating, counts, distr


In [ ]:
# ----------------------------
# Helper: choose marked targets around llm_pred (±radius)
# ----------------------------
def make_marked_indices(llm_pred, radius=0):
    """Return list of indices to mark. radius=0 -> only rounded(llm_pred).
       radius=1 -> round(llm_pred) ± 1 within bounds, etc."""
    center = rating_to_index(llm_pred)
    idxs = []
    for d in range(-radius, radius + 1):
        idx = center + d
        if 0 <= idx < NUM_RATINGS:
            idxs.append(idx)
    # ensure unique
    return sorted(set(idxs))


In [ ]:
# ----------------------------
# Hybrid prediction per row
# ----------------------------
def hybrid_predict_row(row, qnode_maker, grover_iters=1, llm_weight=0.6, radius_mark=0):
    llm_pred, llm_text = llm_predict_row(row)
    # form marked set
    marked_idxs = make_marked_indices(llm_pred, radius=radius_mark)
    # we will mark all these indices in the oracle (multi-target marking by marking each target in sequence)
    # Implementation strategy: in the qnode we call oracle corresponding to each target sequentially before diffusion,
    # but easier: run Grover separately for each marked idx and pick most confident? To keep simple & stable:
    # We'll mark only the rounded prediction (radius=0) OR if radius>0 we'll run Grover with each target and average.
    if len(marked_idxs) == 1:
        target_idx = marked_idxs[0]
        target_bits = index_to_bits(target_idx, n_qubits=N_QUBITS)
         # 🔹 Choisir automatiquement le nombre optimal d’itérations

        optimal_iters = optimal_grover_iterations(N_QUBITS, len(marked_idxs))
        qnode = qnode_maker(optimal_iters)
        # unpack 3 éléments
        q_pred, counts, probs = run_grover_get_rating(qnode, target_bits, k_iter=optimal_iters)
        # result = run_grover_get_rating(qnode, target_bits, k_iter=optimal_iters)
        # print(result)
        # qnode = qnode_maker(grover_iters)
        # q_pred, counts = run_grover_get_rating(qnode, target_bits, k_iter=grover_iters)
    else:
        # run Grover for each marked index and choose the most frequently returned rating
        votes = Counter()
        counts_across = {}
        for idx in marked_idxs:
            bits = index_to_bits(idx, n_qubits=N_QUBITS)
            qnode = qnode_maker(grover_iters)
            q_pred_tmp, counts_tmp = run_grover_get_rating(qnode, bits, k_iter=grover_iters)
            votes[q_pred_tmp] += 1
            counts_across[idx] = counts_tmp
        q_pred = votes.most_common(1)[0][0]
        counts = counts_across
    # combine
    final = llm_weight * float(llm_pred) + (1.0 - llm_weight) * float(q_pred)
    return {
        "llm_pred": float(llm_pred),
        "llm_text": llm_text,
        "q_pred": float(q_pred),
        "counts": counts,
        "final": float(final)
    }


In [ ]:
!nvidia-smi  

In [ ]:

# ----------------------------
# Full evaluation on dataset (RMSE / MAE)
# ----------------------------
def evaluate_on_dataframe(df, label_col="rating", test_size=0.2, random_state=42,
                          grover_iters=1, llm_weight=0.6, radius_mark=0, max_rows=None):
    # optional: limit rows for quick tests
    if max_rows:
        df = df.iloc[:max_rows].copy()
    # basic cleaning: ensure label exists
    df = df.dropna(subset=[label_col])
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state)
    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
    qnode_maker = build_grover_qnode(N_QUBITS, shots=QL_SHOTS)
    preds = []
    trues = []
    for i, row in test_df.iterrows():
        # res = hybrid_predict_row(row, qnode_maker, grover_iters=grover_iters, llm_weight=llm_weight, radius_mark=radius_mark)
        row_dict = row.to_dict()   # <-- conversion
        res = hybrid_predict_row(row_dict, qnode_maker, grover_iters=1, llm_weight=0.6, radius_mark=0)
        preds.append(res['final'])
        trues.append(float(row[label_col]))
        if i % 10 == 0:
            print(f"Idx {i}: llm {res['llm_pred']:.3f}, q {res['q_pred']:.1f}, final {res['final']:.3f}, true {row[label_col]}")
    rmse = math.sqrt(mean_squared_error(trues, preds))
    mae = mean_absolute_error(trues, preds)
    return {"rmse": rmse, "mae": mae, "n": len(trues)}


In [ ]:
!pip install openpyxl

In [ ]:
# import csv
# import os
# import pandas as pd
# import numpy as np
# from tqdm import tqdm
# from sklearn.metrics import mean_absolute_error, mean_squared_error

# # ----------------------------
# # Fonctions auxiliaires
# # ----------------------------
# def compute_metrics(df_partial):
#     """Calcule MAE et RMSE sur les résultats accumulés."""
#     if len(df_partial) == 0:
#         return {"MAE": None, "RMSE": None}
#     mae = mean_absolute_error(df_partial["true_rating"], df_partial["pred_rating"])
#     mse = mean_squared_error(df_partial["true_rating"], df_partial["pred_rating"])
#     rmse = np.sqrt(mse)
#     return {"MAE": mae, "RMSE": rmse}


# def save_partial_results(results_list, output_file):
#     """Sauvegarde partielle avec métriques."""
#     df_partial = pd.DataFrame(results_list)
#     metrics = compute_metrics(df_partial)
#     df_partial["MAE_partial"] = metrics["MAE"]
#     df_partial["RMSE_partial"] = metrics["RMSE"]
#     df_partial.to_csv(output_file, index=False, mode='w', encoding='utf-8')


# # ----------------------------
# # Evaluation principale
# # ----------------------------
# if __name__ == "__main__":
#     csv_path = "all_reviews_filtered.csv"  # ton dataset
#     output_file = "partial_results.csv"    # fichier de sauvegarde partielle

#     # Charger le dataset
#     try:
#         df = pd.read_csv(
#             csv_path,
#             engine='python',
#             quoting=csv.QUOTE_ALL,
#             on_bad_lines='skip',
#             encoding='utf-8'
#         )
#         print("Nombre de lignes chargées:", len(df))
#     except Exception as e:
#         print(f"Error reading {csv_path}: {e}")
#         print("Creating a small toy DataFrame for demo...")
#         df = pd.DataFrame([
#             {"user_id": "user1", "business_id": "biz1", "text": "The food was delicious!", "stars": 5},
#             {"user_id": "user2", "business_id": "biz2", "text": "Average place.", "stars": 3},
#             {"user_id": "user3", "business_id": "biz3", "text": "Terrible service.", "stars": 1},
#         ])

#     # Charger les résultats déjà existants si présents
#     if os.path.exists(output_file):
#         partial_results = pd.read_csv(output_file).to_dict("records")
#         print(f"Reprise à partir de {len(partial_results)} exemples sauvegardés.")
#     else:
#         partial_results = []

#     # Définir l'index de départ pour reprendre
#     start_idx = len(partial_results)

#     # Boucle d’évaluation ligne par ligne
#     for idx, row in tqdm(enumerate(df.iterrows()), total=len(df), desc="Évaluation en cours"):
#         if idx < start_idx:
#             continue  # saute les lignes déjà traitées

#         # --- Ici tu appelles ton modèle LLM ---
#         true_rating = row[1]["stars"]
#         # Exemple : simulation de prédiction (à remplacer par ton LLM) saloua
#         # predicted_rating = np.clip(true_rating + np.random.randn() * 0.3, 1, 5)
#         predicted_rating, decoded_output = llm_predict_row(row[1], max_new_tokens=20)


#         # Ajouter les résultats partiels
#         partial_results.append({
#             "index": idx,
#             "user_id": row[1]["user_id"],
#             "business_id": row[1]["business_id"],
#             "true_rating": true_rating,
#             "pred_rating": predicted_rating
#         })

#         # Sauvegarde toutes les 1000 itérations (moins de sorties)
#         if idx % 1000 == 0 and idx > 0:
#             save_partial_results(partial_results, output_file)
#             metrics = compute_metrics(pd.DataFrame(partial_results))
#             print(f"[Checkpoint @ {idx}] MAE={metrics['MAE']:.4f}, RMSE={metrics['RMSE']:.4f}")

#     # Sauvegarde finale après boucle
#     save_partial_results(partial_results, output_file)
#     metrics = compute_metrics(pd.DataFrame(partial_results))
#     print("✅ Évaluation terminée.")
#     print(f"MAE={metrics['MAE']:.4f}, RMSE={metrics['RMSE']:.4f}")


In [ ]:
# 21 décembre 2025
import math
import numpy as np
import pennylane as qml

# --- Paramètres Grover (comme Amazon) ---
N_RATINGS = 5
N_QUBITS_GROVER = 3
N_SHOTS_GROVER = 200

dev_grover = qml.device("default.qubit", wires=N_QUBITS_GROVER, shots=N_SHOTS_GROVER)

def int_to_bits(x, n_bits):
    return [int(b) for b in format(int(x), f"0{n_bits}b")]

def grover_oracle(target_index):
    bits = int_to_bits(target_index, N_QUBITS_GROVER)

    # amener l'état cible vers |111>
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    # revenir au basis standard
    for i, b in enumerate(bits):
        if b == 0:
            qml.PauliX(wires=i)

def grover_diffuser():
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)
        qml.PauliX(wires=i)

    control_wires = list(range(N_QUBITS_GROVER - 1))
    target_wire = N_QUBITS_GROVER - 1
    qml.MultiControlledX(control_wires=control_wires, wires=target_wire)

    for i in range(N_QUBITS_GROVER):
        qml.PauliX(wires=i)
        qml.Hadamard(wires=i)

def grover_iterations(N, M=1):
    return max(1, round((math.pi / 4.0) * math.sqrt(N / M)))

@qml.qnode(dev_grover)
def grover_circuit(target_index, n_iterations):
    # superposition uniforme
    for i in range(N_QUBITS_GROVER):
        qml.Hadamard(wires=i)

    for _ in range(int(n_iterations)):
        grover_oracle(target_index)
        grover_diffuser()

    return qml.sample(wires=range(N_QUBITS_GROVER))

def run_grover_for_rating(prior_rating: int, k=1):
    """
    prior_rating : rating entier (1..5) (ex: llm_pred)
    k           : nb itérations grover (1 recommandé comme toi)
    retourne    : rating entier (1..5)
    """
    target_index = int(prior_rating) - 1
    target_index = max(0, min(N_RATINGS - 1, target_index))

    samples = grover_circuit(target_index, k)
    samples = np.array(samples)

    # bits -> index entier
    indices = []
    for row in samples:
        v = 0
        for b in row:
            v = (v << 1) | int(b)
        indices.append(v)

    indices = np.array(indices)
    indices = indices[indices < N_RATINGS]  # garder seulement 0..4

    if len(indices) == 0:
        return int(prior_rating)

    values, counts = np.unique(indices, return_counts=True)
    best_index = values[np.argmax(counts)]
    return int(best_index) + 1  # 0..4 -> 1..5


In [ ]:
# 21 décembre 2025
import pandas as pd
import numpy as np

df_results = pd.DataFrame(partial_results)

# --- arrays ---
y_true  = df_results["true_rating"].astype(int).values
y_llm   = df_results["llm_pred"].astype(int).values
y_quant = df_results["q_pred"].astype(int).values
y_qllm  = df_results["pred_rating"].astype(int).values  # final (hybrid)

# --- Grover-only basé sur la sortie LLM ---
# (même définition que Amazon : Grover(prior = llm_pred))
y_grover = np.array([run_grover_for_rating(r, k=1) for r in y_llm], dtype=int)

print("✅ Yelp arrays:", y_true.shape, y_llm.shape, y_quant.shape, y_qllm.shape, y_grover.shape)
print("Unique Grover-only ratings:", np.unique(y_grover))


In [ ]:
models_yelp = [
    ModelResult("LLM",            y_true, y_llm),
    ModelResult("Quantum kNN",    y_true, y_quant),     # même nom que Amazon pour cohérence figure
    ModelResult("Q-LLM",          y_true, y_qllm),
    ModelResult("Grover-only LLM",y_true, y_grover),
]

analyzer_yelp = YelpRatingAnalyzer(models_yelp)
analyzer_yelp.plot_all()


In [ ]:
# code fonctionnel decembre 2025
import csv
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ----------------------------
# Fonctions auxiliaires
# ----------------------------

def compute_metrics(df_partial):
    """Calcule MAE et RMSE sur les résultats accumulés (compatible avec toutes versions sklearn)."""
    if len(df_partial) == 0:
        return {"MAE": None, "RMSE": None}
    mae = mean_absolute_error(df_partial["true_rating"], df_partial["pred_rating"])
    mse = mean_squared_error(df_partial["true_rating"], df_partial["pred_rating"])  # retourne le MSE
    rmse = np.sqrt(mse)
    return {"MAE": mae, "RMSE": rmse}


def save_partial_results(results_list, output_file):
    """Sauvegarde partielle avec métriques."""
    df_partial = pd.DataFrame(results_list)
    metrics = compute_metrics(df_partial)
    df_partial["MAE_partial"] = metrics["MAE"]
    df_partial["RMSE_partial"] = metrics["RMSE"]
    df_partial.to_csv(output_file, index=False, mode='w', encoding='utf-8')

# ----------------------------
# Évaluation principale
# ----------------------------
# if __name__ == "__main__":
#     csv_path = "all_reviews_filtered.csv"
#     output_file = "partial_results_sample.csv"

#     # Charger le dataset
#     df = pd.read_csv(
#         csv_path,
#         engine='python',
#         quoting=csv.QUOTE_ALL,
#         on_bad_lines='skip',
#         encoding='utf-8'
#     )
#     print("Nombre total de lignes chargées:", len(df))

#     # ---------------------------
#     # 🟦 ÉCHANTILLONNAGE ICI !
#     # ---------------------------
#     SAMPLE_SIZE = 10000   # modifie à 10000 si tu veux
#     df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

#     print(f"➡️ Évaluation sur un échantillon de {SAMPLE_SIZE} lignes.")

#     partial_results = []

#     for idx, row in tqdm(df.iterrows(), total=len(df), desc="Évaluation (échantillon)"):
        
#         true_rating = row["stars"]

#         # --- Appel LLM à remplacer ici ---
#         # predicted_rating = np.clip(true_rating + np.random.randn() * 0.4, 1, 5)
#         predicted_rating, decoded_output = llm_predict_row(row[1], max_new_tokens=20)
        
#         partial_results.append({
#             "index": idx,
#             "user_id": row["user_id"],
#             "business_id": row["business_id"],
#             "true_rating": true_rating,
#             "pred_rating": predicted_rating
#         })

#         # Sauvegarde toutes les 500 lignes
#         if idx % 500 == 0 and idx > 0:
#             save_partial_results(partial_results, output_file)

#     # Sauvegarde finale après boucle
#     save_partial_results(partial_results, output_file)
#     metrics = compute_metrics(pd.DataFrame(partial_results))
    
#     print("✅ Évaluation terminée.")
#     print(f"MAE={metrics['MAE']:.4f}, RMSE={metrics['RMSE']:.4f}")
# ----------------------------
# Évaluation sur un échantillon avec LLM + Grover
# ----------------------------
if __name__ == "__main__":
    csv_path = "all_reviews_filtered.csv"
    output_file = "partial_results_sample.csv"

    # Charger le dataset
    df = pd.read_csv(
        csv_path,
        engine='python',
        quoting=csv.QUOTE_ALL,
        on_bad_lines='skip',
        encoding='utf-8'
    )
    print("Nombre total de lignes chargées:", len(df))

    # ---------------------------
    # 🟦 ÉCHANTILLONNAGE ICI !
    # ---------------------------
    SAMPLE_SIZE = 100   # modifie si nécessaire
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"➡️ Évaluation sur un échantillon de {SAMPLE_SIZE} lignes.")

    # Créer le qnode pour Grover
    qnode_maker = build_grover_qnode(N_QUBITS, shots=QL_SHOTS)

    partial_results = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Évaluation (échantillon)"):
        row_dict = row.to_dict()  # <-- convertit la Series en dict

        # Appel de la fonction hybride LLM + Grover
        res = hybrid_predict_row(
            row_dict,
            qnode_maker,
            grover_iters=1,
            llm_weight=0.6,
            radius_mark=0
        )

        true_rating = float(row_dict["stars"])
        predicted_rating = res['final']

        partial_results.append({
            "index": idx,
            "user_id": row_dict["user_id"],
            "business_id": row_dict["business_id"],
            "true_rating": true_rating,
            "pred_rating": predicted_rating,
            "llm_pred": res['llm_pred'],
            "q_pred": res['q_pred'],
        })

        # Sauvegarde toutes les 500 lignes
        if idx % 500 == 0 and idx > 0:
            save_partial_results(partial_results, output_file)

    # Sauvegarde finale après boucle
    save_partial_results(partial_results, output_file)
    metrics = compute_metrics(pd.DataFrame(partial_results))

    print("✅ Évaluation terminée.")
    print(f"MAE={metrics['MAE']:.4f}, RMSE={metrics['RMSE']:.4f}")



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, confusion_matrix

@dataclass
class ModelResult:
    name: str
    y_true: np.ndarray
    y_pred: np.ndarray

class YelpRatingAnalyzer:
    """
    Version simple qui:
    - calcule metrics
    - génère figures
    - SAUVEGARDE automatiquement en PNG (haute qualité)
    """
    def __init__(self, models: List[ModelResult], labels=None, out_dir="figures_qllm_yelp"):
        self.models = models
        self.y_true = models[0].y_true.astype(int)
        self.labels = labels if labels is not None else np.arange(1, 6)
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)

    def compute_metrics(self) -> pd.DataFrame:
        rows = []
        for m in self.models:
            y_pred = m.y_pred.astype(int)
            rows.append({
                "Model": m.name,
                "Accuracy": accuracy_score(self.y_true, y_pred),
                "MAE": mean_absolute_error(self.y_true, y_pred),
                "RMSE": np.sqrt(mean_squared_error(self.y_true, y_pred)),
            })
        return pd.DataFrame(rows).set_index("Model")

    def _save(self, filename):
        path = os.path.join(self.out_dir, filename)
        plt.tight_layout()
        plt.savefig(path, dpi=300, bbox_inches="tight")
        plt.close()
        print("✅ Saved:", path)

    def plot_metrics_bar(self):
        df = self.compute_metrics().reset_index()

        plt.figure(figsize=(9, 3.2))
        for i, metric in enumerate(["Accuracy", "MAE", "RMSE"], start=1):
            plt.subplot(1, 3, i)
            plt.bar(df["Model"], df[metric])
            plt.title(metric)
            plt.xticks(rotation=20, ha="right")
            plt.grid(alpha=0.3, axis="y")
            if metric == "Accuracy":
                plt.ylim(0, 1)

        self._save("00_metrics_bar.png")

    def plot_true_vs_pred_scatter(self):
        plt.figure(figsize=(6.5, 4.2))
        for m in self.models:
            plt.scatter(self.y_true, m.y_pred.astype(int), s=35, alpha=0.55, label=m.name)

        plt.plot([self.labels.min(), self.labels.max()],
                 [self.labels.min(), self.labels.max()], "k--", alpha=0.7)

        plt.xticks(self.labels)
        plt.yticks(self.labels)
        plt.xlabel("True Rating")
        plt.ylabel("Predicted Rating")
        plt.title("True vs Predicted (Yelp)")
        plt.grid(alpha=0.3)
        plt.legend()
        self._save("01_true_vs_pred_scatter.png")

    def plot_confusion_matrices(self, selected=("LLM", "Q-LLM")):
        name_to_model = {m.name: m for m in self.models}
        selected = [n for n in selected if n in name_to_model]
        if len(selected) == 0:
            print("⚠️ Aucun modèle trouvé pour confusion matrices:", selected)
            return

        fig, axes = plt.subplots(1, len(selected), figsize=(5.8 * len(selected), 4.6))
        if len(selected) == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            m = name_to_model[name]
            cm = confusion_matrix(self.y_true, m.y_pred.astype(int), labels=self.labels)
            im = ax.imshow(cm, interpolation="nearest")
            ax.set_title(f"Confusion Matrix - {name}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_xticks(range(len(self.labels)))
            ax.set_yticks(range(len(self.labels)))
            ax.set_xticklabels(self.labels)
            ax.set_yticklabels(self.labels)

            # annotations
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9)

        plt.colorbar(im, ax=axes, fraction=0.03, pad=0.02)
        self._save("02_confusion_matrices.png")

    def plot_error_histograms(self):
        plt.figure(figsize=(7.2, 4.2))
        bins = np.arange(-4.5, 4.6, 1)

        for m in self.models:
            err = m.y_pred.astype(int) - self.y_true
            plt.hist(err, bins=bins, alpha=0.45, density=True, label=m.name)

        plt.xlabel("Error = y_pred - y_true")
        plt.ylabel("Density")
        plt.title("Error Distribution (Yelp)")
        plt.grid(alpha=0.3)
        plt.legend()
        self._save("03_error_histograms.png")

    def plot_abs_error_boxplot(self):
        data = []
        labels = []
        for m in self.models:
            abs_err = np.abs(m.y_pred.astype(int) - self.y_true)
            data.append(abs_err)
            labels.append(m.name)

        plt.figure(figsize=(7.0, 4.0))
        plt.boxplot(data, labels=labels, showfliers=True)
        plt.ylabel("|y_pred - y_true|")
        plt.title("Absolute Error (Yelp)")
        plt.grid(alpha=0.3, axis="y")
        self._save("04_abs_error_boxplot.png")

    def plot_quantum_advantage_box(self, baseline="LLM", target="Q-LLM"):
        name_to_model = {m.name: m for m in self.models}
        if baseline not in name_to_model or target not in name_to_model:
            print(f"⚠️ Missing {baseline} or {target} for advantage plot.")
            return

        base = np.abs(name_to_model[baseline].y_pred.astype(int) - self.y_true)
        tgt  = np.abs(name_to_model[target].y_pred.astype(int) - self.y_true)
        adv = base - tgt

        plt.figure(figsize=(4.2, 4.2))
        plt.boxplot([adv], labels=["Δ advantage"])
        # plt reminder = plt.axhline(0.0, linestyle="--", alpha=0.7)
        plt.axhline(0.0, linestyle="--", alpha=0.7)

        plt.ylabel("|err_LLM| - |err_Q-LLM|")
        plt.title("Quantum Advantage (Yelp)")
        plt.grid(alpha=0.3, axis="y")
        self._save("05_quantum_advantage.png")

    def plot_all(self):
        print("=== Yelp Metrics ===")
        print(self.compute_metrics())
        self.plot_metrics_bar()
        self.plot_true_vs_pred_scatter()
        self.plot_confusion_matrices(selected=("LLM","Q-LLM"))
        self.plot_error_histograms()
        self.plot_abs_error_boxplot()
        self.plot_quantum_advantage_box(baseline="LLM", target="Q-LLM")


In [ ]:
df_results = pd.DataFrame(partial_results)

y_true  = df_results["true_rating"].values.astype(int)
y_llm   = df_results["llm_pred"].values.astype(int)
y_quant = df_results["q_pred"].values.astype(int)
y_qllm  = df_results["pred_rating"].values.astype(int)   # final hybrid

print("✅ Yelp arrays ready:", y_true.shape)

models_yelp = [
    ModelResult("LLM",     y_true, y_llm),
    ModelResult("Quantum", y_true, y_quant),
    ModelResult("Q-LLM",   y_true, y_qllm),
]

analyzer_yelp = YelpRatingAnalyzer(models_yelp, out_dir="figures_qllm_yelp")
analyzer_yelp.plot_all()


In [ ]:
# code pour générer les figures december 2025
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1) Construire df_results
# ----------------------------
assert "partial_results" in globals() and len(partial_results) > 0, \
    "❌ partial_results est vide ou non défini. Exécute d'abord la boucle d'évaluation Yelp."

df_results = pd.DataFrame(partial_results)

required_cols = {"true_rating", "pred_rating", "llm_pred", "q_pred"}
missing = required_cols - set(df_results.columns)
assert len(missing) == 0, f"❌ Colonnes manquantes dans df_results: {missing}"

# Vecteurs numpy
y_true  = df_results["true_rating"].astype(int).values
y_qllm  = df_results["pred_rating"].astype(int).values   # Hybrid final = Q-LLM
y_llm   = df_results["llm_pred"].astype(int).values
y_quant = df_results["q_pred"].astype(int).values

print("✅ Yelp arrays ready:", y_true.shape)

# ----------------------------
# 2) Construire les modèles (noms cohérents papier)
# ----------------------------
models_yelp = [
    ModelResult("LLM",     y_true, y_llm),
    ModelResult("Quantum", y_true, y_quant),
    ModelResult("Q-LLM",   y_true, y_qllm),
]

# ----------------------------
# 3) Option A: utiliser TON analyzer (si tu as YelpRatingAnalyzer)
#    + sauver les figures en surchargeant plt.show()
# ----------------------------
OUT_DIR = "figures_qllm_yelp"
os.makedirs(OUT_DIR, exist_ok=True)

# Hack simple : chaque plt.show() => savefig automatique
_fig_id = 0
_original_show = plt.show

def _save_show():
    global _fig_id
    path = os.path.join(OUT_DIR, f"yelp_{_fig_id:02d}.png")
    plt.gcf().savefig(path, dpi=300, bbox_inches="tight")
    print("💾 Saved:", path)
    _fig_id += 1
    _original_show()

plt.show = _save_show

# Lancer l'analyse
analyzer_yelp = YelpRatingAnalyzer(models_yelp)  # ou RatingPredictionAnalyzer(...) selon ton notebook
analyzer_yelp.plot_all()

# Restaurer show normal
plt.show = _original_show

print(f"✅ Figures Yelp sauvegardées dans: {OUT_DIR}")


In [ ]:
    # On convertit en DataFrame pour l’analyse
df_results = pd.DataFrame(partial_results)

# Vecteurs numpy
y_true = df_results["true_rating"].values.astype(float)
y_final = df_results["pred_rating"].values.astype(float)    # Hybrid LLM + Grover
y_llm = df_results["llm_pred"].values.astype(float)         # prédiction LLM seule
y_q = df_results["q_pred"].values.astype(float)             # prédiction quantique seule

models_yelp = [
    ModelResult("LLM",     y_true, y_llm),
    ModelResult("Quantum", y_true, y_q),
    ModelResult("Hybrid",  y_true, y_final),
]

analyzer_yelp = YelpRatingAnalyzer(models_yelp)
analyzer_yelp.plot_all()

In [ ]:
!nvidia-smi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, confusion_matrix

plt.style.use("default")
sns.set_palette("husl")
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 14,
})


@dataclass
class ModelResult:
    name: str
    y_true: np.ndarray
    y_pred: np.ndarray


class YelpRatingAnalyzer:
    """
    Analyseur pour Yelp :
    - calcule Accuracy / MAE / RMSE
    - génère différentes figures comparant LLM, Quantum, Hybride
    """

    def __init__(self, models: List[ModelResult]):
        self.models = models
        self.y_true = models[0].y_true
        self.labels = np.arange(1, 6)

    def compute_metrics(self) -> pd.DataFrame:
        rows = []
        for m in self.models:
            mae = mean_absolute_error(m.y_true, m.y_pred)
            rmse = np.sqrt(mean_squared_error(m.y_true, m.y_pred))
            acc = accuracy_score(m.y_true, np.round(m.y_pred).astype(int))
            rows.append({
                "Model": m.name,
                "Accuracy": acc,
                "MAE": mae,
                "RMSE": rmse,
            })
        df_metrics = pd.DataFrame(rows).set_index("Model")
        return df_metrics

    def plot_metrics_bar(self):
        df = self.compute_metrics().reset_index()

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        sns.barplot(data=df, x="Model", y="Accuracy", ax=axes[0])
        axes[0].set_title("Accuracy")
        axes[0].set_ylim(0, 1)
        axes[0].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="MAE", ax=axes[1])
        axes[1].set_title("MAE")
        axes[1].grid(alpha=0.3, axis="y")

        sns.barplot(data=df, x="Model", y="RMSE", ax=axes[2])
        axes[2].set_title("RMSE")
        axes[2].grid(alpha=0.3, axis="y")

        for ax in axes:
            for label in ax.get_xticklabels():
                label.set_rotation(20)
                label.set_ha("right")

        plt.tight_layout()
        plt.show()

    def plot_error_histograms(self):
        plt.figure(figsize=(10, 6))
        for m in self.models:
            errors = m.y_pred - self.y_true
            sns.histplot(errors, kde=True, stat="density", label=m.name, bins=np.arange(-4.5, 4.6, 0.5), alpha=0.5)
        plt.xlabel("Error = y_pred - y_true")
        plt.ylabel("Density")
        plt.title("Error Distribution per Model (Yelp)")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_abs_error_boxplot(self):
        data = []
        for m in self.models:
            abs_err = np.abs(m.y_pred - self.y_true)
            data.append(pd.DataFrame({
                "Model": m.name,
                "Absolute Error": abs_err
            }))
        df = pd.concat(data, ignore_index=True)

        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df, x="Model", y="Absolute Error")
        plt.title("Absolute Error Distribution per Model (Yelp)")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    def plot_true_vs_pred_scatter(self):
        name_to_model = {m.name: m for m in self.models}
        main_names = [n for n in ["LLM", "Quantum", "Hybrid"] if n in name_to_model]

        plt.figure(figsize=(10, 6))
        for name in main_names:
            m = name_to_model[name]
            plt.scatter(self.y_true, m.y_pred, alpha=0.4, s=40, label=name)

        plt.plot([1, 5], [1, 5], "k--", alpha=0.7)
        plt.xlim(0.5, 5.5)
        plt.ylim(0.5, 5.5)
        plt.xticks([1, 2, 3, 4, 5])
        plt.yticks([1, 2, 3, 4, 5])
        plt.xlabel("True Rating")
        plt.ylabel("Predicted Rating")
        plt.title("True vs Predicted Ratings (Yelp)")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_confusion_matrices(self):
        name_to_model = {m.name: m for m in self.models}
        selected = [n for n in ["LLM", "Hybrid"] if n in name_to_model]

        if not selected:
            print("No LLM/Hybrid models found for confusion matrices.")
            return

        fig, axes = plt.subplots(1, len(selected), figsize=(6 * len(selected), 5))
        if len(selected) == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            m = name_to_model[name]
            # on arrondit les prédictions pour la matrice de confusion
            cm = confusion_matrix(self.y_true, np.round(m.y_pred).astype(int), labels=self.labels)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=self.labels, yticklabels=self.labels, ax=ax)
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_title(f"Confusion Matrix - {name} (Yelp)")

        plt.tight_layout()
        plt.show()

    def plot_quantum_advantage_box(self, baseline_name="LLM", target_name="Hybrid"):
        name_to_model = {m.name: m for m in self.models}
        if baseline_name not in name_to_model or target_name not in name_to_model:
            print(f"Missing {baseline_name} or {target_name} for advantage plot.")
            return

        m_base = name_to_model[baseline_name]
        m_tgt = name_to_model[target_name]

        base_abs = np.abs(m_base.y_pred - self.y_true)
        tgt_abs = np.abs(m_tgt.y_pred - self.y_true)
        advantage = base_abs - tgt_abs  # > 0 => Hybrid meilleur

        plt.figure(figsize=(6, 5))
        sns.boxplot(data=pd.DataFrame({"Quantum Advantage (Yelp)": advantage}))
        plt.axhline(0.0, color="red", linestyle="--", alpha=0.7)
        plt.title(f"Quantum Advantage (Yelp): |err_{baseline_name}| - |err_{target_name}|")
        plt.ylabel("Advantage (>0 => target better)")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    def plot_all(self):
        print("=== Yelp Metrics Table ===")
        df_metrics = self.compute_metrics()
        try:
            display(df_metrics)  # Jupyter
        except NameError:
            print(df_metrics)

        self.plot_metrics_bar()
        self.plot_true_vs_pred_scatter()
        self.plot_confusion_matrices()
        self.plot_error_histograms()
        self.plot_abs_error_boxplot()
        self.plot_quantum_advantage_box(baseline_name="LLM", target_name="Hybrid")


In [ ]:
# import csv 
# # ----------------------------
# # Example usage
# # ----------------------------
# if __name__ == "__main__":
#     # Path to your Yelp CSV file
#     csv_path = "all_reviews_filtered.csv"  # <-- change to your real path

#     try:
#         # df = pd.read_csv(csv_path)
#         # df = pd.read_csv(
#         #     "all_reviews_filtered.csv",
#         #     quotechar='"',
#         #     escapechar='\\',
#         #     on_bad_lines='skip',
#         #     encoding='utf-8'
#         # )
#         df = pd.read_csv(
#             "all_reviews_filtered.csv",
#             engine='python',         # plus tolérant
#             quoting=csv.QUOTE_ALL,   # prend en compte les guillemets
#             on_bad_lines='skip',     # saute les lignes malformées
#             encoding='utf-8'
#         )

#         print("Nombre de lignes chargées:", len(df))
#     except Exception as e:
#         print(f"Error reading {csv_path}: {e}")
#         print("Creating a small toy DataFrame for demo...")
#         df = pd.DataFrame([
#             {
#                 "user_id": "user1",
#                 "business_id": "biz1",
#                 "text": "The food was delicious and the service was great!",
#                 "stars": 5
#             },
#             {
#                 "user_id": "user2",
#                 "business_id": "biz2",
#                 "text": "Average experience, nothing special about this place.",
#                 "stars": 3
#             },
#             {
#                 "user_id": "user3",
#                 "business_id": "biz3",
#                 "text": "Terrible service and cold food. Not recommended.",
#                 "stars": 1
#             },
#         ])

#     # Quick evaluate (set max_rows small if you want fast execution)
#     # metrics = evaluate_on_dataframe(df, label_col="rating", test_size=0.5,
#     #                                 grover_iters=1, llm_weight=0.6, radius_mark=0, max_rows=10)
#     metrics = evaluate_on_dataframe(
#         df,
#         label_col="stars",       # target column in Yelp dataset
#         test_size=0.5,
#         grover_iters=1,
#         llm_weight=0.6,
#         radius_mark=0,
#         max_rows=None              # reduce for quick test
#     )

#     print("Evaluation:", metrics)
